In [ ]:
import Pkg
Pkg.add("DataFrames")
Pkg.add("CSV")
Pkg.add("XLSX")
Pkg.add("Statistics")
Pkg.add("Turing")
Pkg.add("DifferentialEquations")
Pkg.add("StatsPlots")
using DataFrames, CSV, XLSX, Statistics

In [ ]:
using Turing
using DifferentialEquations
using StatsPlots
using LinearAlgebra
import NaNMath

In [ ]:
using DataFrames, XLSX

In [ ]:
med4 = DataFrame(XLSX.readtable("PSSP7_ZT145.xlsx", "Host"))
rename!(med4, [:time, :rep1, :rep2])
t_obs_pro = med4.time
obsdata_pro = (med4.rep1 .+ med4.rep2) ./ 2

virus = DataFrame(XLSX.readtable("PSSP7_ZT145.xlsx", "Virus"))
rename!(virus, [:time, :rep1, :rep2])
t_obs_virus = virus.time
obsdata_virus = (virus.rep1 .+ virus.rep2) ./ 2
nothing

In [ ]:
function pro_virus_basic(du, u, p, t)
    # state variables
    Su = u[1] # susceptible host cells
    Ex = u[2] # exposed host cells
    In = u[3] # infected host cells and the capsid-protected progeny phages appear
    Vi = u[4] # virus
    

    # parameters
    μmax = p[1] # maximum growth rate at Lopt (h⁻¹)
    Lopt = p[2] # optimal light (μmol s⁻¹ m⁻²)
    α    = p[3] # initial slope of the light response curve (h⁻¹)
    KL   = p[4] # minimum amount of light necessary for cell division (μmol s⁻¹ m⁻²)
    ω    = p[5] # hose basal mortality (h⁻¹)
    K    = p[6] # host carrying capacity (cells ml⁻¹)
    ϕ    = p[7] # adsorption rate (ml h⁻¹)
    β    = p[8] # burst size (unitless)
    λe   = p[9] # average eclise period (h)(the time between phage attachemnt and the appearence of capsid-protected phages)
    λl   = p[10] # average lysis period (h)(the time needed to lyse host cells after the appearence of capsid-protected phages)
    δ    = p[11] # viral decay rate (h⁻¹)
    
    # light -- hourly data
    n_light = 14; n_dark = 10
    L = 35 # μmol s⁻¹ m⁻²
    # t in hours
    τ = rem(t, 24) # time of day
    
    # light dependent growth rate
    μopt = μmax * L / (L + μmax / α * (L / Lopt - 1.0)^2)
    Lt = L * τ * (1.0 - isless(n_light, τ)) + L * (n_light - (τ - n_dark)) * n_light / n_dark * isless(n_light, τ)
    μ = μopt * Lt^4 / (Lt^4 + KL^4)
   

    total_cells = Su + Ex + In 
    carrying_capacity_factor = max(1.0 - total_cells/K, 0.0)
    

    du[1] = μ * Su * carrying_capacity_factor - ω * Su - ϕ * Su * Vi           # dSu/dt
    du[2] = ϕ * Su * Vi - ω * Ex - Ex / λe                                     # dEx/dt
    du[3] = Ex / λe - ω * In - In / λl                                     # dIn/dt 
    du[4] = β * In / λl -  ϕ * Su * Vi - δ * Vi                                 # dVi/dt
   
   return nothing
end

#export resistance_ratio, pro_virus_basic 

In [ ]:
# ============================================================
# 从 vPRO 参数文件读取参数并应用到模型
# ============================================================
# 注意：vPRO 参数只包含宿主生长相关参数，需要补充病毒相关参数

# 读取 vPRO 参数文件
# 注意：如果文件路径不对，请根据实际情况修改
# 可能的路径选项：
# - "../V4/vpro_parameters.csv" (如果 notebook 在 V3 目录下)
# - "vpro_parameters.csv" (如果文件在同一目录)
# - 绝对路径
vpro_params_file = "../V4/vpro_parameters.csv"

# 如果文件不存在，尝试其他路径
if !isfile(vpro_params_file)
    # 尝试当前目录
    alt_path = "vpro_parameters.csv"
    if isfile(alt_path)
        vpro_params_file = alt_path
    else
        println("警告: 找不到 vPRO 参数文件，请检查路径: $vpro_params_file")
    end
end

vpro_df = CSV.read(vpro_params_file, DataFrame)

println("=" ^ 70)
println("📋 读取 vPRO 参数")
println("=" ^ 70)
println("成功读取 $(nrow(vpro_df)) 组 vPRO 参数")
println("\n参数列: $(names(vpro_df))")

# ============================================================
# 设置缺失参数的默认值（病毒相关参数）
# ============================================================
# 使用原始模型中的病毒参数作为默认值
# 可以根据需要修改这些值
DEFAULT_VIRUS_PARAMS = Dict(
    :K => 2.8e9,        # 环境容纳量 (cells/mL) - 可以使用 CSV 中的 init 值或固定值
    :ϕ => 2.0e-9,       # 吸附速率 (mL h⁻¹)
    :β => 200.0,        # 裂解量
    :λe => 35.0,        # 潜伏期 (h)
    :λl => 30.0,        # 裂解期 (h)
    :δ => 6.0e-4        # 病毒衰减率 (h⁻¹)
)

println("\n使用默认病毒参数：")
for (key, val) in DEFAULT_VIRUS_PARAMS
    println("  $key = $val")
end

# ============================================================
# 将 vPRO 参数转换为模型参数格式
# ============================================================
# 模型参数顺序: [μmax, Lopt, α, KL, ω, K, ϕ, β, λe, λl, δ]
function vpro_to_model_params(row)
    # 从 CSV 行提取 vPRO 参数
    μmax = row.μmax
    Lopt = row.Lopt
    α = row.α
    KL = row.KL
    ω = row.ω
    
    # 使用 init 作为环境容纳量 K（或使用默认值）
    # 注意：init 可能是初始细胞数量，通常比环境容纳量 K 小
    # 这里使用固定默认值（推荐），也可以根据 init 计算 K
    # 如果要用 init 相关的值，可以尝试: K = row.init * 10 等
    K = DEFAULT_VIRUS_PARAMS[:K]  # 使用固定默认值
    # 或者: K = row.init * 10  # 如果 init 是初始值，可能需要放大
    
    # 添加病毒相关参数
    ϕ = DEFAULT_VIRUS_PARAMS[:ϕ]
    β = DEFAULT_VIRUS_PARAMS[:β]
    λe = DEFAULT_VIRUS_PARAMS[:λe]
    λl = DEFAULT_VIRUS_PARAMS[:λl]
    δ = DEFAULT_VIRUS_PARAMS[:δ]
    
    return [μmax, Lopt, α, KL, ω, K, ϕ, β, λe, λl, δ]
end

# 转换所有 vPRO 参数
model_params_list = [vpro_to_model_params(row) for row in eachrow(vpro_df)]

println("\n" * "=" ^ 70)
println("✅ 参数转换完成")
println("=" ^ 70)
println("共转换 $(length(model_params_list)) 组参数")

# 显示第一组参数示例
println("\n第一组参数示例 (vPRO ID: $(vpro_df.vpro_id[1])):")
param_names_model = [:μmax, :Lopt, :α, :KL, :ω, :K, :ϕ, :β, :λe, :λl, :δ]
for (i, name) in enumerate(param_names_model)
    println("  $name = $(round(model_params_list[1][i], sigdigits=4))")
end

In [ ]:
# ============================================================
# 使用 vPRO 参数运行模型并评估结果
# ============================================================

# 初始条件和时间范围
u0_vpro = [obsdata_pro[1], obsdata_pro[1]*1e-6, obsdata_pro[1]*1e-6, obsdata_virus[1]]
tspan_vpro = (14.0, 136.0)

# 存储所有解和评估结果
vpro_solutions = []
vpro_successful_params = []
vpro_fit_scores = []
vpro_failed_ids = []
vpro_failed_reasons = Dict{Int, Any}()  # 存储失败原因

println("\n" * "=" ^ 70)
println("🚀 开始使用 vPRO 参数求解模型")
println("=" ^ 70)
println("初始条件: Su=$(u0_vpro[1]), Ex=$(u0_vpro[2]), In=$(u0_vpro[3]), Vi=$(u0_vpro[4])")
println("时间范围: $(tspan_vpro[1]) - $(tspan_vpro[2]) h\n")

# 先测试第一组参数，看具体错误
println("测试第一组参数 (vPRO ID: $(vpro_df.vpro_id[1]))...")
test_p = model_params_list[1]
test_prob = ODEProblem(pro_virus_basic, u0_vpro, tspan_vpro, test_p)
try
    test_sol = solve(test_prob, Tsit5(); 
                saveat=0.1,
                abstol=1e-8, 
                reltol=1e-8,
                maxiters=1e7)
    println("  求解状态: $(test_sol.retcode)")
    # 在 Julia 中，成功可能表示为 :Success 符号或 "Success" 字符串
    if test_sol.retcode == :Success || test_sol.retcode == Symbol("Success") || string(test_sol.retcode) == "Success"
        println("  求解成功！")
    else
        println("  失败原因: $(test_sol.retcode)")
    end
catch e
    println("  错误: $e")
end
println()

for (i, p) in enumerate(model_params_list)
    vpro_id = vpro_df.vpro_id[i]
    prob = ODEProblem(pro_virus_basic, u0_vpro, tspan_vpro, p)
    
    try
        # 尝试多种求解器设置
        sol = nothing
        success_flag = false
        
        # 方法1: 标准设置
        try
            sol = solve(prob, Tsit5(); 
                        saveat=0.1,
                        abstol=1e-8, 
                        reltol=1e-8,
                        maxiters=1e7)
            # 检查是否成功 (可能是符号 :Success 或字符串 "Success")
            if sol.retcode == :Success || sol.retcode == Symbol("Success") || string(sol.retcode) == "Success"
                success_flag = true
            end
        catch
        end
        
        # 方法2: 更宽松的容差
        if !success_flag
            try
                sol = solve(prob, Tsit5(); 
                            saveat=0.1,
                            abstol=1e-6, 
                            reltol=1e-6,
                            maxiters=1e7)
                if sol.retcode == :Success || (isdefined(Main, :ReturnCode) && sol.retcode == ReturnCode.Success)
                    success_flag = true
                end
            catch
            end
        end
        
        # 方法3: 使用更宽松的容差和更粗的时间步
        if !success_flag
            try
                sol = solve(prob, Tsit5(); 
                            saveat=1.0,  # 更大的时间步
                            abstol=1e-4, 
                            reltol=1e-4,
                            maxiters=1e7)
                if sol.retcode == :Success || (isdefined(Main, :ReturnCode) && sol.retcode == ReturnCode.Success)
                    success_flag = true
                end
            catch
            end
        end
        
        # 方法4: 使用 RK4 求解器（更稳定）
        if !success_flag
            try
                sol = solve(prob, RK4(); 
                            saveat=0.5,
                            dt=0.01,
                            maxiters=1e7)
                if sol.retcode == :Success || (isdefined(Main, :ReturnCode) && sol.retcode == ReturnCode.Success)
                    success_flag = true
                end
            catch
            end
        end
        
        # 最终检查：如果成功就处理结果
        if success_flag && (sol.retcode == :Success || sol.retcode == Symbol("Success") || string(sol.retcode) == "Success")
            # 评估拟合优度
            fit_result = evaluate_fit(sol, t_obs_pro, obsdata_pro, t_obs_virus, obsdata_virus)
            
            push!(vpro_solutions, sol)
            push!(vpro_successful_params, p)
            push!(vpro_fit_scores, fit_result)
        else
            push!(vpro_failed_ids, vpro_id)
            if sol !== nothing
                # 记录实际的返回代码
                vpro_failed_reasons[vpro_id] = "返回代码: $(sol.retcode)"
            else
                vpro_failed_reasons[vpro_id] = "求解器异常"
            end
        end
    catch e
        push!(vpro_failed_ids, vpro_id)
        vpro_failed_reasons[vpro_id] = "异常: $(typeof(e))"
    end
    
    # 进度显示
    if i % 10 == 0 || i == length(model_params_list)
        println("  已处理: $i / $(length(model_params_list)) (vPRO ID: $vpro_id) | 成功: $(length(vpro_solutions))")
    end
end

println("\n" * "=" ^ 70)
println("📊 求解结果统计")
println("=" ^ 70)
println("  总参数组数: $(length(model_params_list))")
println("  成功求解: $(length(vpro_solutions))")
println("  失败数量: $(length(vpro_failed_ids))")
if length(vpro_failed_ids) > 0
    println("\n失败的 vPRO ID 和原因（前10个）:")
    failed_shown = 0
    for (vpro_id, reason) in vpro_failed_reasons
        if failed_shown < 10
            println("  vPRO ID $vpro_id: $reason")
            failed_shown += 1
        end
    end
    if length(vpro_failed_ids) > 10
        println("  ... 还有 $(length(vpro_failed_ids) - 10) 个失败")
    end
end

# ============================================================
# 找到最佳拟合参数
# ============================================================
if length(vpro_fit_scores) > 0
    # 按综合评分排序
    sorted_indices_vpro = sortperm([s.total_score for s in vpro_fit_scores])
    best_idx_vpro = sorted_indices_vpro[1]
    
    # 找到对应的 vPRO ID
    successful_vpro_ids = [vpro_df.vpro_id[i] for i in 1:nrow(vpro_df) if !(vpro_df.vpro_id[i] in vpro_failed_ids)]
    best_vpro_id = successful_vpro_ids[best_idx_vpro]
    
    best_params_vpro = vpro_successful_params[best_idx_vpro]
    best_score_vpro = vpro_fit_scores[best_idx_vpro]
    
    println("\n" * "=" ^ 70)
    println("🏆 最佳 vPRO 参数组合")
    println("=" ^ 70)
    println("vPRO ID: $best_vpro_id")
    println("\n参数值：")
    for (i, name) in enumerate(param_names_model)
        println("  $(rpad(string(name), 6)) = $(round(best_params_vpro[i], sigdigits=4))")
    end
    println("\n拟合优度：")
    println("  R² (宿主):    $(round(best_score_vpro.r2_host, digits=4))")
    println("  R² (病毒):    $(round(best_score_vpro.r2_virus, digits=4))")
    println("  NRMSE (宿主): $(round(best_score_vpro.nrmse_host, digits=4))")
    println("  NRMSE (病毒): $(round(best_score_vpro.nrmse_virus, digits=4))")
    println("  综合评分:     $(round(best_score_vpro.total_score, digits=4))")
    
    # 显示前 10 个最佳参数
    println("\n" * "=" ^ 70)
    println("📈 Top 10 最佳 vPRO 参数")
    println("=" ^ 70)
    println(rpad("排名", 6), rpad("vPRO ID", 10), rpad("R²(宿主)", 12), rpad("R²(病毒)", 12), rpad("NRMSE(宿主)", 14), rpad("综合评分", 10))
    println("-" ^ 70)
    for rank in 1:min(10, length(sorted_indices_vpro))
        idx = sorted_indices_vpro[rank]
        vpro_id = successful_vpro_ids[idx]
        score = vpro_fit_scores[idx]
        println(rpad(rank, 6), 
                rpad(vpro_id, 10),
                rpad(round(score.r2_host, digits=4), 12),
                rpad(round(score.r2_virus, digits=4), 12),
                rpad(round(score.nrmse_host, digits=4), 14),
                rpad(round(score.total_score, digits=4), 10))
    end
end

In [ ]:
# ============================================================
# 可视化 vPRO 参数拟合结果
# ============================================================

if length(vpro_solutions) > 0
    # 创建宿主和病毒的子图
    p1_vpro = plot(
        xlabel="Time (h)",
        ylabel="Cell count (cells/mL)",
        legend=:outertopright,
        left_margin=8Plots.mm,
        bottom_margin=6Plots.mm,
        title="Host Cells - vPRO Parameters ($(length(vpro_solutions)) sets)"
    )
    
    p2_vpro = plot(
        xlabel="Time (h)",
        ylabel="Virus count (copies/mL)",
        yscale=:log10,
        legend=:outertopright,
        left_margin=8Plots.mm,
        bottom_margin=6Plots.mm,
        title="Virus - vPRO Parameters ($(length(vpro_solutions)) sets)"
    )
    
    # 绘制所有成功的拟合曲线
    for (idx, sol) in enumerate(vpro_solutions)
        sol_array = Array(sol)'
        total_host = sol_array[:,1] .+ sol_array[:,2] .+ sol_array[:,3]
        plot!(p1_vpro, sol.t, total_host, alpha=0.2, color=:blue, linewidth=0.5, label="")
        plot!(p2_vpro, sol.t, sol_array[:,4], alpha=0.2, color=:red, linewidth=0.5, label="")
    end
    
    # 绘制最佳拟合曲线（加粗）
    if length(vpro_fit_scores) > 0
        best_sol_vpro = vpro_solutions[best_idx_vpro]
        best_array_vpro = Array(best_sol_vpro)'
        best_host_vpro = best_array_vpro[:,1] .+ best_array_vpro[:,2] .+ best_array_vpro[:,3]
        
        plot!(p1_vpro, best_sol_vpro.t, best_host_vpro, 
              linewidth=3, color=:darkblue, label="Best fit (vPRO $(best_vpro_id))")
        plot!(p2_vpro, best_sol_vpro.t, best_array_vpro[:,4], 
              linewidth=3, color=:darkred, label="Best fit (vPRO $(best_vpro_id))")
    end
    
    # 添加观测数据点
    scatter!(p1_vpro, t_obs_pro, obsdata_pro, label="Host data", color=:black, markersize=6)
    scatter!(p2_vpro, t_obs_virus, obsdata_virus, label="Virus data", color=:black, markersize=6)
    
    # 组合图
    plot_vpro = plot(p1_vpro, p2_vpro, size=(1200, 500), layout=(1, 2))
    display(plot_vpro)
    
    # ============================================================
    # 保存结果
    # ============================================================
    using Dates
    ts_vpro = Dates.format(now(), "yyyymmdd_HHMMSS")
    
    # 保存图片
    savefig(plot_vpro, "vpro_fit_results_$(ts_vpro).pdf")
    println("\n✅ 图片已保存: vpro_fit_results_$(ts_vpro).pdf")
    
    # 保存所有 vPRO 参数的拟合结果到 CSV
    successful_vpro_ids_all = [vpro_df.vpro_id[i] for i in 1:nrow(vpro_df) if !(vpro_df.vpro_id[i] in vpro_failed_ids)]
    
    vpro_results_df = DataFrame()
    vpro_results_df[!, :vpro_id] = successful_vpro_ids_all
    vpro_results_df[!, :μmax] = [p[1] for p in vpro_successful_params]
    vpro_results_df[!, :Lopt] = [p[2] for p in vpro_successful_params]
    vpro_results_df[!, :α] = [p[3] for p in vpro_successful_params]
    vpro_results_df[!, :KL] = [p[4] for p in vpro_successful_params]
    vpro_results_df[!, :ω] = [p[5] for p in vpro_successful_params]
    vpro_results_df[!, :K] = [p[6] for p in vpro_successful_params]
    vpro_results_df[!, :R2_host] = [s.r2_host for s in vpro_fit_scores]
    vpro_results_df[!, :R2_virus] = [s.r2_virus for s in vpro_fit_scores]
    vpro_results_df[!, :NRMSE_host] = [s.nrmse_host for s in vpro_fit_scores]
    vpro_results_df[!, :NRMSE_virus] = [s.nrmse_virus for s in vpro_fit_scores]
    vpro_results_df[!, :total_score] = [s.total_score for s in vpro_fit_scores]
    
    # 按综合评分排序
    sort!(vpro_results_df, :total_score)
    CSV.write("vpro_model_results_$(ts_vpro).csv", vpro_results_df)
    println("✅ 结果已保存: vpro_model_results_$(ts_vpro).csv")
    
    println("\n" * "=" ^ 70)
    println("📁 保存的文件")
    println("=" ^ 70)
    println("  vpro_fit_results_$(ts_vpro).pdf      - 拟合可视化图")
    println("  vpro_model_results_$(ts_vpro).csv    - 所有 vPRO 参数的拟合结果")
    println("\n保存目录: $(pwd())")
else
    println("❌ 没有成功求解的参数组合，无法生成可视化")
end

In [ ]:
using Random
Random.seed!(42)  # 设置随机种子，保证可重复性

# ============================================================
# 首先测试原始参数是否能成功求解
# ============================================================
println("=" ^ 60)
println("测试原始参数...")
u0_test = [obsdata_pro[1], obsdata_pro[1]*1e-6, obsdata_pro[1]*1e-6, obsdata_virus[1]]
p_original = [0.025, 45.78, 6.1e-4, 255.0, 0.0015, 2.8e9, 0.2e-8, 200.0, 35.0, 30.0, 6e-4]
tspan_test = (14.0, 136.0)
prob_test = ODEProblem(pro_virus_basic, u0_test, tspan_test, p_original)
sol_test = solve(prob_test, Tsit5(); saveat=1.0, abstol=1e-8, reltol=1e-8)
println("原始参数求解状态: $(sol_test.retcode)")
println("=" ^ 60)

# ============================================================
# 定义参数范围 (基于原始参数的 ±200% 范围，扩大搜索空间)
# ============================================================
# 原始参数: [0.025, 45.78, 6.1e-4, 255.0, 0.0015, 2.8e9, 0.2e-8, 200.0, 35, 30, 6e-4]
param_ranges = Dict(
    :μmax => (0.00833, 0.075),      # 最大生长率 (h⁻¹) - 原值 0.025 (±200%)
    :Lopt => (15.26, 137.34),       # 最适光强 - 原值 45.78 (±200%)
    :α    => (2.03e-4, 1.83e-3),    # 光响应曲线初始斜率 - 原值 6.1e-4 (±200%)
    :KL   => (85.0, 765.0),         # 最低光强 - 原值 255.0 (±200%)
    :ω    => (0.0005, 0.0045),      # 宿主基础死亡率 - 原值 0.0015 (±200%)
    :K    => (9.33e8, 8.4e9),       # 环境容纳量 - 原值 2.8e9 (±200%)
    :ϕ    => (6.67e-10, 6e-9),      # 吸附速率 - 原值 2e-9 (0.2e-8) (±200%)
    :β    => (66.67, 600.0),        # 裂解量 - 原值 200 (±200%)
    :λe   => (11.67, 105.0),        # 潜伏期 - 原值 35 (±200%)
    :λl   => (10.0, 90.0),          # 裂解期 - 原值 30 (±200%)
    :δ    => (2e-4, 1.8e-3)         # 病毒衰减率 - 原值 6e-4 (±200%)
)

# 参数名称顺序（与模型中 p 向量的顺序一致）
param_names = [:μmax, :Lopt, :α, :KL, :ω, :K, :ϕ, :β, :λe, :λl, :δ]

# ============================================================
# 生成随机参数组合
# ============================================================
n_samples = 10000  # 生成 10000 组随机参数

# 从均匀分布中随机采样
function sample_parameters(ranges, names, n)
    params = []
    for i in 1:n
        p = [rand() * (ranges[name][2] - ranges[name][1]) + ranges[name][1] for name in names]
        push!(params, p)
    end
    return params
end

# 生成参数集合
param_sets = sample_parameters(param_ranges, param_names, n_samples)

println("生成了 $(n_samples) 组随机参数组合")
println("\n参数范围：")
for name in param_names
    println("  $(name): $(param_ranges[name])")
end

# 显示前 5 组参数示例
println("\n前 5 组参数示例：")
for i in 1:min(5, n_samples)
    println("  参数组 $i: ", round.(param_sets[i], sigdigits=3))
end

# ============================================================
# 初始条件和时间范围
# ============================================================
u0 = [obsdata_pro[1], obsdata_pro[1]*1e-6, obsdata_pro[1]*1e-6, obsdata_virus[1]]
tspan = (14.0, 136.0)

# ============================================================
# 定义拟合优度评估函数
# ============================================================
"""
计算模型预测与观测数据的拟合优度
返回: (RMSE_host, RMSE_virus, R²_host, R²_virus, total_score)
"""
function evaluate_fit(sol, t_obs_host, obs_host, t_obs_virus, obs_virus)
    # 在观测时间点插值获取模型预测值
    # 宿主：总细胞数 = Su + Ex + In
    pred_host = [sol(t)[1] + sol(t)[2] + sol(t)[3] for t in t_obs_host]
    # 病毒
    pred_virus = [sol(t)[4] for t in t_obs_virus]
    
    # 计算 RMSE (均方根误差)
    rmse_host = sqrt(mean((pred_host .- obs_host).^2))
    rmse_virus_log = sqrt(mean((log10.(max.(pred_virus, 1.0)) .- log10.(max.(obs_virus, 1.0))).^2))
    
    # 计算 R² (决定系数)
    ss_res_host = sum((pred_host .- obs_host).^2)
    ss_tot_host = sum((obs_host .- mean(obs_host)).^2)
    r2_host = 1 - ss_res_host / ss_tot_host
    
    ss_res_virus = sum((log10.(max.(pred_virus, 1.0)) .- log10.(max.(obs_virus, 1.0))).^2)
    ss_tot_virus = sum((log10.(max.(obs_virus, 1.0)) .- mean(log10.(max.(obs_virus, 1.0)))).^2)
    r2_virus = 1 - ss_res_virus / ss_tot_virus
    
    # 归一化 RMSE (相对于观测数据的标准差)
    nrmse_host = rmse_host / std(obs_host)
    nrmse_virus = rmse_virus_log / std(log10.(max.(obs_virus, 1.0)))
    
    # 综合评分 (越小越好)
    # 可以调整权重来平衡宿主和病毒的重要性
    total_score = 0.5 * nrmse_host + 0.5 * nrmse_virus
    
    return (rmse_host=rmse_host, rmse_virus=rmse_virus_log, 
            r2_host=r2_host, r2_virus=r2_virus,
            nrmse_host=nrmse_host, nrmse_virus=nrmse_virus,
            total_score=total_score)
end

# ============================================================
# 设置筛选标准
# ============================================================
# 方法1: 基于 R² 阈值
R2_THRESHOLD_HOST = 0.5      # 宿主 R² 至少 0.5
R2_THRESHOLD_VIRUS = 0.5     # 病毒 R² 至少 0.5

# 方法2: 基于 NRMSE 阈值 (归一化均方根误差)
NRMSE_THRESHOLD = 1.0        # NRMSE < 1 表示误差小于数据标准差

# 方法3: 保留最好的 N% 参数组合
TOP_PERCENT = 10             # 保留最好的 10%

println("\n筛选标准：")
println("  R² 阈值 (宿主): ≥ $(R2_THRESHOLD_HOST)")
println("  R² 阈值 (病毒): ≥ $(R2_THRESHOLD_VIRUS)")
println("  NRMSE 阈值: ≤ $(NRMSE_THRESHOLD)")
println("  保留最好的: $(TOP_PERCENT)%")

# ============================================================
# 对所有参数组合求解 ODE 并评估
# ============================================================
all_solutions = []
successful_params = []
fit_scores = []
failed_reasons = Dict{Symbol, Int}()

println("\n开始求解和评估...")

for (i, p) in enumerate(param_sets)
    prob = ODEProblem(pro_virus_basic, u0, tspan, p)
    try
        sol = solve(prob, Tsit5(); 
                    saveat=0.1,
                    abstol=1e-6, 
                    reltol=1e-6,
                    maxiters=1e7)
        
        if sol.retcode == ReturnCode.Success
            # 评估拟合优度
            fit_result = evaluate_fit(sol, t_obs_pro, obsdata_pro, t_obs_virus, obsdata_virus)
            
            push!(all_solutions, sol)
            push!(successful_params, p)
            push!(fit_scores, fit_result)
        else
            reason = Symbol(sol.retcode)
            failed_reasons[reason] = get(failed_reasons, reason, 0) + 1
        end
    catch e
        failed_reasons[:Exception] = get(failed_reasons, :Exception, 0) + 1
    end
    
    # 进度显示
    if i % 1000 == 0
        println("  已处理: $i / $n_samples")
    end
end

println("\n成功求解: $(length(all_solutions)) / $(n_samples) 组参数")

# ============================================================
# 根据标准筛选参数
# ============================================================
# 方法1: R² 阈值筛选
good_by_r2 = findall(s -> s.r2_host >= R2_THRESHOLD_HOST && s.r2_virus >= R2_THRESHOLD_VIRUS, fit_scores)

# 方法2: NRMSE 阈值筛选
good_by_nrmse = findall(s -> s.nrmse_host <= NRMSE_THRESHOLD && s.nrmse_virus <= NRMSE_THRESHOLD, fit_scores)

# 方法3: 保留最好的 N%
n_top = max(1, Int(ceil(length(fit_scores) * TOP_PERCENT / 100)))
sorted_indices = sortperm([s.total_score for s in fit_scores])
good_by_top = sorted_indices[1:n_top]

# 综合筛选：满足所有条件
good_indices = intersect(good_by_r2, good_by_nrmse)

println("\n筛选结果：")
println("  通过 R² 阈值: $(length(good_by_r2)) 组")
println("  通过 NRMSE 阈值: $(length(good_by_nrmse)) 组")
println("  最好的 $(TOP_PERCENT)%: $(length(good_by_top)) 组")
println("  综合通过: $(length(good_indices)) 组")

# ============================================================
# 显示最佳参数组合
# ============================================================
if length(fit_scores) > 0
    best_idx = sorted_indices[1]
    best_params = successful_params[best_idx]
    best_score = fit_scores[best_idx]
    
    println("\n" * "=" ^ 60)
    println("最佳参数组合：")
    println("=" ^ 60)
    for (i, name) in enumerate(param_names)
        println("  $(name) = $(round(best_params[i], sigdigits=4))")
    end
    println("\n拟合优度：")
    println("  R² (宿主): $(round(best_score.r2_host, digits=4))")
    println("  R² (病毒): $(round(best_score.r2_virus, digits=4))")
    println("  NRMSE (宿主): $(round(best_score.nrmse_host, digits=4))")
    println("  NRMSE (病毒): $(round(best_score.nrmse_virus, digits=4))")
    println("  综合评分: $(round(best_score.total_score, digits=4))")
end

# ============================================================
# 绑图：显示筛选后的结果
# ============================================================
# 使用所有通过筛选标准的参数组合绑图
plot_indices = good_indices

p1 = plot(
    xlabel="Time (h)",
    ylabel="Cell count (cells/mL)",
    legend=:outertopright,
    left_margin=8Plots.mm,
    bottom_margin=6Plots.mm,
    title="Host Cells - All Passed ($(length(plot_indices)) sets)"
)

p2 = plot(
    xlabel="Time (h)",
    ylabel="Virus count (copies/mL)",
    yscale=:log10,
    legend=:outertopright,
    left_margin=8Plots.mm,
    bottom_margin=6Plots.mm,
    title="Virus - All Passed ($(length(plot_indices)) sets)"
)

# 绑制筛选后的曲线
for idx in plot_indices
    sol = all_solutions[idx]
    sol_array = Array(sol)'
    total_host = sol_array[:,1] .+ sol_array[:,2] .+ sol_array[:,3]
    plot!(p1, sol.t, total_host, alpha=0.3, color=:blue, label="")
    plot!(p2, sol.t, sol_array[:,4], alpha=0.3, color=:red, label="")
end

# 绑制最佳拟合曲线（加粗）
if length(fit_scores) > 0
    best_sol = all_solutions[best_idx]
    best_array = Array(best_sol)'
    best_host = best_array[:,1] .+ best_array[:,2] .+ best_array[:,3]
    plot!(p1, best_sol.t, best_host, linewidth=3, color=:darkblue, label="Best fit")
    plot!(p2, best_sol.t, best_array[:,4], linewidth=3, color=:darkred, label="Best fit")
end

# 添加观测数据点
scatter!(p1, t_obs_pro, obsdata_pro, label="Host data", color=:black, markersize=6)
scatter!(p2, t_obs_virus, obsdata_virus, label="Virus data", color=:black, markersize=6)

# 组合图
plot(p1, p2, size=(1200, 500), layout=(1, 2))

In [ ]:
# ============================================================
# 保存结果到文件 - 保留所有通过筛选标准的参数组合
# ============================================================
using Dates

ts = Dates.format(now(), "yyyymmdd_HHMMSS")

# ============================================================
# 计算通过比例
# ============================================================
n_total = n_samples                           # 总采样数
n_solved = length(all_solutions)              # 成功求解数
n_pass_r2 = length(good_by_r2)                # 通过 R² 阈值
n_pass_nrmse = length(good_by_nrmse)          # 通过 NRMSE 阈值
n_pass_all = length(good_indices)             # 综合通过（同时满足 R² 和 NRMSE）

# 计算比例
ratio_solved = n_solved / n_total * 100
ratio_pass_r2 = n_pass_r2 / n_solved * 100
ratio_pass_nrmse = n_pass_nrmse / n_solved * 100
ratio_pass_all = n_pass_all / n_solved * 100
ratio_pass_all_total = n_pass_all / n_total * 100

println("=" ^ 70)
println("📊 筛选结果统计")
println("=" ^ 70)
println("\n采样与求解：")
println("  总采样数:        $(n_total)")
println("  成功求解:        $(n_solved) ($(round(ratio_solved, digits=2))%)")

println("\n筛选通过统计：")
println("  通过 R² 阈值:    $(n_pass_r2) / $(n_solved) = $(round(ratio_pass_r2, digits=2))%")
println("  通过 NRMSE 阈值: $(n_pass_nrmse) / $(n_solved) = $(round(ratio_pass_nrmse, digits=2))%")
println("  综合通过:        $(n_pass_all) / $(n_solved) = $(round(ratio_pass_all, digits=2))%")
println("  总通过率:        $(n_pass_all) / $(n_total) = $(round(ratio_pass_all_total, digits=2))%")

# ============================================================
# 1. 保存最佳参数到 CSV
# ============================================================
best_params_df = DataFrame(
    Parameter = String.(param_names),
    Value = best_params,
    Description = [
        "最大生长率 (h⁻¹)",
        "最适光强 (μmol s⁻¹ m⁻²)",
        "光响应曲线初始斜率",
        "最低光强 (μmol s⁻¹ m⁻²)",
        "宿主基础死亡率 (h⁻¹)",
        "环境容纳量 (cells/mL)",
        "吸附速率 (mL h⁻¹)",
        "裂解量",
        "潜伏期 (h)",
        "裂解期 (h)",
        "病毒衰减率 (h⁻¹)"
    ]
)
CSV.write("best_params_$(ts).csv", best_params_df)

# ============================================================
# 2. 保存所有通过筛选标准的参数组合
# ============================================================
# 使用综合通过的参数（同时满足 R² 和 NRMSE 阈值）
passed_params_df = DataFrame()
for (i, name) in enumerate(param_names)
    passed_params_df[!, name] = [successful_params[idx][i] for idx in good_indices]
end
# 添加拟合优度指标
passed_params_df[!, :R2_host] = [fit_scores[idx].r2_host for idx in good_indices]
passed_params_df[!, :R2_virus] = [fit_scores[idx].r2_virus for idx in good_indices]
passed_params_df[!, :NRMSE_host] = [fit_scores[idx].nrmse_host for idx in good_indices]
passed_params_df[!, :NRMSE_virus] = [fit_scores[idx].nrmse_virus for idx in good_indices]
passed_params_df[!, :total_score] = [fit_scores[idx].total_score for idx in good_indices]

# 按综合评分排序
sort!(passed_params_df, :total_score)
CSV.write("passed_params_all_$(ts).csv", passed_params_df)

# ============================================================
# 3. 计算通过参数的统计摘要（均值、标准差、范围）
# ============================================================
println("\n" * "=" ^ 70)
println("📈 通过筛选参数的统计摘要")
println("=" ^ 70)
println("\n参数统计 ($(n_pass_all) 组通过筛选)：")
println("-" ^ 70)
println(rpad("参数", 10), rpad("均值", 15), rpad("标准差", 15), rpad("最小值", 15), "最大值")
println("-" ^ 70)
for name in param_names
    vals = passed_params_df[!, name]
    m = mean(vals)
    s = std(vals)
    mi = minimum(vals)
    ma = maximum(vals)
    println(rpad(string(name), 10), 
            rpad(string(round(m, sigdigits=4)), 15),
            rpad(string(round(s, sigdigits=4)), 15),
            rpad(string(round(mi, sigdigits=4)), 15),
            string(round(ma, sigdigits=4)))
end

# ============================================================
# 4. 保存图片
# ============================================================
savefig(p1, "figure_host_passed_$(ts).pdf")
savefig(p2, "figure_virus_passed_$(ts).pdf")

# ============================================================
# 5. 显示结果摘要
# ============================================================
println("\n" * "=" ^ 70)
println("📊 最佳参数组合")
println("=" ^ 70)
for (i, name) in enumerate(param_names)
    println("  $(rpad(string(name), 8)) = $(best_params[i])")
end
println("\n拟合优度：")
println("  R² (宿主):    $(round(best_score.r2_host, digits=4)) (解释了 $(round(best_score.r2_host*100, digits=1))% 的数据变异)")
println("  R² (病毒):    $(round(best_score.r2_virus, digits=4)) (解释了 $(round(best_score.r2_virus*100, digits=1))% 的数据变异)")
println("  NRMSE (宿主): $(round(best_score.nrmse_host, digits=4)) (误差为数据标准差的 $(round(best_score.nrmse_host*100, digits=1))%)")
println("  NRMSE (病毒): $(round(best_score.nrmse_virus, digits=4)) (误差为数据标准差的 $(round(best_score.nrmse_virus*100, digits=1))%)")

println("\n" * "=" ^ 70)
println("📁 保存的文件")
println("=" ^ 70)
println("  best_params_$(ts).csv         - 最佳参数 (1 组)")
println("  passed_params_all_$(ts).csv   - 所有通过筛选的参数 ($(n_pass_all) 组)")
println("  figure_host_passed_$(ts).pdf  - 宿主拟合图")
println("  figure_virus_passed_$(ts).pdf - 病毒拟合图")
println("\n保存目录: $(pwd())")
